# Project 10: Online Shopping Behaviour - Analysing Customer Activity and Predicting Purchase Intent

**Author:** IDRA Data Science & AI Capstone

---

## Phase 1: Data Understanding & Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

df = pd.read_csv('../data/P_10_Ecommerce_Cleaned.csv')
print('Dataset Shape:', df.shape)
df.head()

## Phase 2: Data Cleaning & Type Conversion

In [ ]:
# Check missing values
print('Missing values:')
print(df.isnull().sum())

# Date parsing & temporal feature extraction
df['visit_date_dt'] = pd.to_datetime(df['visit_date'], format='%d-%m-%Y')
df['visit_year'] = df['visit_date_dt'].dt.year
df['visit_day_of_year'] = df['visit_date_dt'].dt.dayofyear
print('Data types after date parsing:')
print(df.dtypes)

## Phase 3: Exploratory Data Analysis (EDA)

In [ ]:
print('Target Class Distribution (purchased):')
print(df['purchased'].value_counts(normalize=True))

# Group comparisons
print('\nPurchase rate by User Type:')
print(df.groupby('user_type')['purchased'].mean())

print('\nPurchase rate by Device Type:')
print(df.groupby('device_type')['purchased'].mean())

print('\nPurchase rate by Marketing Channel:')
print(df.groupby('marketing_channel')['purchased'].mean())

## Phase 4: Feature Selection & Leakage Prevention

Excluding target (`purchased`) and outcome leakage variables (`revenue`, `revenue_normalized`, `cart_abandoned`).

In [ ]:
features = [
    'customer_id', 'session_id', 'device_type', 'user_type', 'marketing_channel',
    'product_id', 'product_category', 'unit_price', 'quantity', 'discount_percent',
    'discount_amount', 'pages_viewed', 'time_on_site_sec', 'added_to_cart', 'rating',
    'review_text', 'review_helpful_votes', 'payment_method', 'visit_day', 'visit_month',
    'visit_weekday', 'visit_season', 'session_duration_bucket', 'location',
    'visit_year', 'visit_day_of_year'
]

X = df[features]
y = df['purchased']
print('Features count:', len(features))

## Phase 5: Train / Test Split & Pipeline Architecture

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = [c for c in features if c != 'session_duration_bucket']
cat_cols = ['session_duration_bucket']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=8, random_state=42))
])

pipeline.fit(X_train, y_train)
print('Pipeline fitted successfully!')

## Phase 6 & 7: Model Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred))
print('Recall:', recall_score(y_test, y_pred))
print('F1-Score:', f1_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_prob))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

## Phase 8: Model Packaging Verification

In [ ]:
# Verify authorative model file
saved_model = joblib.load('../model/P10_purchase_intent_model.joblib')
saved_pred = saved_model.predict(X_test)
print('Authoritative Saved Model Test Accuracy:', accuracy_score(y_test, saved_pred))